In [ ]:
!pip install kaggle

In [ ]:
import os
import json

kaggle_username = "akhi1205"
kaggle_key = "KGAT_0527bc3897e94be0ec5482501d763c71"

kaggle_json = {
    "username": kaggle_username,
    "key": kaggle_key
}

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_json, f)

os.chmod("/root/.kaggle/kaggle.json", 600)

print("Kaggle API configured successfully!")

Kaggle API configured successfully!


In [ ]:
!kaggle datasets download -d ggrill/foodseg103

Dataset URL: https://www.kaggle.com/datasets/ggrill/foodseg103
License(s): apache-2.0
 97% 1.14G/1.17G [00:11<00:00, 43.6MB/s]
100% 1.17G/1.17G [00:11<00:00, 107MB/s] 


In [ ]:
import zipfile

with zipfile.ZipFile("foodseg103.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/foodseg103")

print("Dataset extracted!")

Dataset extracted!


In [ ]:
base_path = "/content/foodseg103/FoodSeg103"

image_dir = f"{base_path}/Images/img_dir"
mask_dir = f"{base_path}/Images/ann_dir"

train_txt = f"{base_path}/ImageSets/train.txt"
val_txt   = f"{base_path}/ImageSets/test.txt"

In [ ]:
def load_ids(txt_path):
    with open(txt_path, "r") as f:
        ids = f.read().splitlines()
    return ids

train_ids = load_ids(train_txt)
val_ids   = load_ids(val_txt)

print("Train samples:", len(train_ids))
print("Validation samples:", len(val_ids))

Train samples: 4983
Validation samples: 2135


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

shutil.copy(
    "/content/foodseg103/FoodSeg103/category_id.txt",
    "/content/drive/MyDrive/category_id.txt"
)

print("Category file saved to Drive.")

Mounted at /content/drive
Category file saved to Drive.


In [ ]:
import cv2
import torch
import os
from torch.utils.data import Dataset

class FoodSegDataset(Dataset):
    def __init__(self, ids, image_dir, mask_dir, split):
        self.ids = ids
        self.image_dir = os.path.join(image_dir, split)
        self.mask_dir = os.path.join(mask_dir, split)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_name = self.ids[idx]

        img_path = os.path.join(self.image_dir, img_name)
        mask_name = img_name.replace(".jpg", ".png")
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, 0)

        # 🔥 RESIZE BOTH
        image = cv2.resize(image, (256, 256))
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)

        image = image / 255.0

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(mask, dtype=torch.long)

        return image, mask

In [ ]:
from torch.utils.data import DataLoader

train_dataset = FoodSegDataset(train_ids, image_dir, mask_dir, split="train")
val_dataset   = FoodSegDataset(val_ids, image_dir, mask_dir, split="test")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

In [ ]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.1 MB/s eta 0:00:00


In [ ]:
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=104
)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [ ]:
model.load_state_dict(
    torch.load("/content/drive/MyDrive/foodseg_best_model.pth", map_location=device)
)

model.to(device)
model.eval()

DeepLabV3Plus(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=Tr

In [ ]:
import torch.nn as nn
import torch

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

num_epochs = 40

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, masks)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")

    if (epoch + 1) % 5 == 0:
      torch.save(
        model.state_dict(),
        f"/content/drive/MyDrive/foodseg_epoch_{epoch+1}.pth"
    )

Epoch 1/40, Loss: 241.1383
Epoch 2/40, Loss: 234.6652
Epoch 3/40, Loss: 229.1471
Epoch 4/40, Loss: 221.3585
Epoch 5/40, Loss: 219.5836
Epoch 6/40, Loss: 214.4571
Epoch 7/40, Loss: 212.2581
Epoch 8/40, Loss: 206.7784
Epoch 9/40, Loss: 203.9262
Epoch 10/40, Loss: 197.2581
Epoch 11/40, Loss: 195.5321
Epoch 12/40, Loss: 192.1593
Epoch 13/40, Loss: 188.6962
Epoch 14/40, Loss: 184.6082
Epoch 15/40, Loss: 181.8633
Epoch 16/40, Loss: 177.7431
Epoch 17/40, Loss: 175.5840
Epoch 18/40, Loss: 172.9278
Epoch 19/40, Loss: 171.3010
Epoch 20/40, Loss: 168.7406
Epoch 21/40, Loss: 166.8291
Epoch 22/40, Loss: 164.2329
Epoch 23/40, Loss: 162.7209
Epoch 24/40, Loss: 159.9861
Epoch 25/40, Loss: 157.5986
Epoch 26/40, Loss: 155.6912
Epoch 27/40, Loss: 153.5633
Epoch 28/40, Loss: 152.2816
Epoch 29/40, Loss: 150.2046
Epoch 30/40, Loss: 148.0973
Epoch 31/40, Loss: 145.5584
Epoch 32/40, Loss: 145.7251
Epoch 33/40, Loss: 144.0370
Epoch 34/40, Loss: 141.3776
Epoch 35/40, Loss: 140.4670
Epoch 36/40, Loss: 138.9580
E

In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/foodseg_best_model_v2.pth"
)